In [1]:
# !pip install pandas pyarrow tabulate langchain langchain_openai langchain_experimental

In [2]:
import langchain

langchain.__version__

'0.3.21'

# Load the data

In [ ]:
import pandas as pd

DATA_FILE = "data/spend_data.parquet"
df = pd.read_parquet(DATA_FILE)
df["delivery_date"] = pd.to_datetime(df["delivery_date"])
df["ordered_date"] = pd.to_datetime(df["ordered_date"])
print(df.head().to_markdown())

|    | order_id                             | department_name   | vendor_name        | item_name   |   item_quantity |   unit_cost |   total_cost | ordered_by       | ordered_date        | approved_by       | delivery_date       |
|---:|:-------------------------------------|:------------------|:-------------------|:------------|----------------:|------------:|-------------:|:-----------------|:--------------------|:------------------|:--------------------|
|  0 | e811cda0-e90c-4553-890f-20762f7f35c2 | Electronics       | TechSuppliers Inc. | Keyboard    |               2 |      341.58 |       683.16 | Jeffrey Lawrence | 2024-08-17 20:10:09 | Debra Gardner     | 2024-08-25 20:10:09 |
|  1 | ec44a8f4-c814-4ec6-958d-e17508b05302 | Maintenance       | OfficeMax Corp     | Tools       |               2 |      117.13 |       234.26 | Maria Montgomery | 2023-04-24 15:19:30 | Christopher Davis | 2023-04-26 15:19:30 |
|  2 | e0ccaef8-de5b-4406-8ad4-0d11af76be46 | Maintenance       | GlobalProc

# Langchain tool calling agent

In [ ]:
from datetime import date
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_experimental.tools import PythonAstREPLTool
from langchain.agents import create_tool_calling_agent, AgentExecutor


today = date.today().strftime("%B %d, %Y")
system_prompt = (
    "Given a pandas dataframe `df` answer user's query."
    "The name of the dataframe is `df`.\n"
    "Here's the output of `df.head().to_markdown()` for your reference, you have access to full dataframe as `df`:\n"
    f"{df.head().to_markdown()}\n\n"
    f"If no day or month or year are mentioned, assume the current day, month and year respectively. Current date is {today}.\n"
    "If the query asks for a report based on time periods, such as daily, monthly, quarterly, annual, "
    "   generate a complete breakdown for each requested period.\n"  
    "Give final answer as soon as you have enough data, otherwise generate code using `df` and call required tool.\n"
    "Query:"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
local_vars = {'pd': pd, 'df': df}
tools = [PythonAstREPLTool(locals=local_vars)]

agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [12]:
# user_query = "Who were the top 3 approvers?"
# user_query = "Which department spent the most in 2023?"
# user_query = "What are the top 3 departments by total spend?"
# user_query = "What is my total monthly spend?"
user_query = "Who were the top 3 vendors for the department that spend the most?"

response = agent_executor.invoke({"input": user_query})['output']



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "top_department = df.groupby('department_name')['total_cost'].sum().idxmax()\ntop_vendors = df[df['department_name'] == top_department].groupby('vendor_name')['total_cost'].sum().nlargest(3)"}`



Invoking: `python_repl_ast` with `{'query': "top_department = df.groupby('department_name')['total_cost'].sum().idxmax()\ntop_vendors = df[df['department_name'] == top_department].groupby('vendor_name')['total_cost'].sum().nlargest(3)\ntop_vendors"}`


vendor_name
OfficeMax Corp    43003.27
MegaVendor        36101.27
ElectroTech       35230.59
Name: total_cost, dtype: float64The top 3 vendors for the department that spent the most are:

1. **OfficeMax Corp** - $43,003.27
2. **MegaVendor** - $36,101.27
3. **ElectroTech** - $35,230.59

> Finished chain.


In [13]:
print(response)

The top 3 vendors for the department that spent the most are:

1. **OfficeMax Corp** - $43,003.27
2. **MegaVendor** - $36,101.27
3. **ElectroTech** - $35,230.59


# Langchain dataframe agent

In [7]:
from langchain_openai import ChatOpenAI
from langchain.agents.agent_types import AgentType
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
agent_df = create_pandas_dataframe_agent(
    llm,
    df,
    verbose=True,
    # agent_type=AgentType.OPENAI_FUNCTIONS,
    agent_type="tool-calling",
    # include_df_in_prompt=True,
    # number_of_head_rows=5,
    allow_dangerous_code=True,
)


In [10]:
# user_query = "Who were the top 3 vendors for the department that spend the most?"
# user_query = "What is my total monthly spend?"
user_query = "Who were the top 3 vendors for the department that spend the most?"
response = agent_df.invoke(user_query)["output"]



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "top_vendors = df.groupby(['department_name', 'vendor_name'])['total_cost'].sum().reset_index()\n\n# Get the department that spent the most\nmax_department = top_vendors.groupby('department_name')['total_cost'].sum().idxmax()\n\n# Filter for the top vendors in that department\ntop_vendors_in_max_department = top_vendors[top_vendors['department_name'] == max_department]\n\n# Get the top 3 vendors by total cost\ntop_3_vendors = top_vendors_in_max_department.nlargest(3, 'total_cost')\n\ntop_3_vendors"}`


  department_name     vendor_name  total_cost
4     Electronics  OfficeMax Corp    43003.27
3     Electronics      MegaVendor    36101.27
0     Electronics     ElectroTech    35230.59The top 3 vendors for the department that spent the most (Electronics) are:

1. **OfficeMax Corp** - Total Cost: $43,003.27
2. **MegaVendor** - Total Cost: $36,101.27
3. **ElectroTech** - Total Cost: $35,230.59

> Finished c

In [11]:
print(response)

The top 3 vendors for the department that spent the most (Electronics) are:

1. **OfficeMax Corp** - Total Cost: $43,003.27
2. **MegaVendor** - Total Cost: $36,101.27
3. **ElectroTech** - Total Cost: $35,230.59
